In [ ]:
from optics_design_workbench.jupyter_utils import *
import sympy as sy
from matplotlib.pyplot import *
from numpy import *

# Ensure that excluded areas are dark if phi domain is set 

In [ ]:
#openFreecadGui( workInTempCopy=False )

In [ ]:
with FreecadDocument( workInTempCopy=True ) as f:

  for distribution in (
    '1',
  ):
    for thetaDomain in (
      '0, .06',
    ):
      # update simulation settings
      f.OpticalSimulationSettings.EndAfterRays = 'inf'
      f.OpticalSimulationSettings.EndAfterHits = '1e5'

      # update source parameters
      s = f.OpticalPointSource
      s.Placement.Angle = 180+20
      s.PowerDensity = '1'
      s.FocalLength = '0'
      s.PowerDensity = distribution
      s.ThetaDomain = thetaDomain
      s.PhiDomain = '0, 3/2*pi'
      s.RaysPerFan = 50
      s.Fans = 12

      rFan = f.runSimulation('fans')
      rTrue = f.runSimulation('true')

In [ ]:
H = rFan.loadHits('*')
for fanI, (theta, powers) in H.fanEstimatedPowerDensities().items():
  plot(theta, powers/8+fanI, label=f'fan #{fanI}')

  if fanI >= 7:
    assert min(theta) >= 0 

legend()

In [ ]:
H = rTrue.loadHits('*').histogram(bins=30)
H.plot()

In [ ]:
assert H.hist[15,20] == 0

In [ ]:
assert H.hist[15,14] > 0

In [ ]:
assert H.hist[19,20] > 0

In [ ]:
assert H.hist[11,20] > 0

# Ensure that astigmatic power density actually yields astigmatic densities

In [ ]:
with FreecadDocument( workInTempCopy=True ) as f:

  for distribution in (
    'exp(-2*((theta*cos(phi))^2/(0.01^2) + (theta*sin(phi))^2/(0.1^2)))',
  ):
    for thetaDomain in (
      '0, .06',
    ):
      # update simulation settings
      f.OpticalSimulationSettings.EndAfterRays = 'inf'
      f.OpticalSimulationSettings.EndAfterHits = '1e5'

      # update source parameters
      s = f.OpticalPointSource
      s.PowerDensity = '1'
      s.FocalLength = '0'
      s.PowerDensity = distribution
      s.ThetaDomain = thetaDomain
      s.PhiDomain = '0, 2*pi'
      s.RaysPerFan = 50
      s.Fans = 12

      rFan = f.runSimulation('fans')
      rTrue = f.runSimulation('true')

In [ ]:
H = rFan.loadHits('*')
for fanI, (theta, powers) in H.fanEstimatedPowerDensities().items():
  plot(theta, powers+fanI, label=f'fan #{fanI}')

  if fanI == 6:
    assert max(powers)-min(powers) < 10 
  if fanI == 0:
    assert max(powers)-min(powers) > 30 

legend()

In [ ]:
(H:=rTrue.loadHits('*').histogram(bins=30)).plot()

In [ ]:
assert H.hist[15,15] > 300

In [ ]:
assert H.hist[15,20] < 200

In [ ]:
assert H.hist[20,20] > 200

In [ ]:
assert H.hist[20,15] < 200

In [ ]:
assert H.hist[10,10] > 300